# Notebook 13 — Front Behavior Models

## Mục tiêu
Huấn luyện và so sánh mô hình dự đoán hành vi từ các đặc trưng quỹ đạo của Notebook 11 và nhãn thủ công từ Notebook 12.

### Nguyên tắc
- Chỉ dùng nhãn `CERTAIN`.
- Loại `UNCERTAIN` và `EXCLUDE`.
- Giảm leakage do cửa sổ 5s/step1s bằng cách chỉ lấy cửa sổ cách nhau 5s cho modeling.
- Cross-validation theo **time block group**, không chia ngẫu nhiên từng window.
- So sánh ít nhất: majority baseline, Logistic Regression, Random Forest.
- Chọn mô hình theo **macro F1**, không chỉ accuracy.
- Export model + feature schema để Notebook 18 triển khai Raspberry Pi.

In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
from pathlib import Path
import json, math, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root():
    candidates = [Path("/home/diy-hus/fish"), Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for p in candidates:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p.resolve()
    return Path("/home/diy-hus/fish").resolve()

PROJECT_ROOT = find_project_root()
FEATURE_PATH = PROJECT_ROOT / "results/behavior/front_individual_behavior_features.csv"
LABEL_PATH = PROJECT_ROOT / "results/behavior/front_behavior_labels.csv"
SCHEMA_PATH = PROJECT_ROOT / "results/behavior/front_behavior_feature_schema.json"

RESULTS_DIR = PROJECT_ROOT / "results/behavior"
MODEL_DIR = PROJECT_ROOT / "models/behavior"
LOG_DIR = PROJECT_ROOT / "logs/behavior/FRONT_BEHAVIOR_MODELS_001"
for p in [RESULTS_DIR, MODEL_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert FEATURE_PATH.exists(), "Run Notebook 11 first."
assert LABEL_PATH.exists(), "Run Notebook 12 first."
assert SCHEMA_PATH.exists(), "Run Notebook 11 first."

In [ ]:
# ============================================================
# 1. CONFIG — FINAL 4-CLASS BEHAVIOR MODEL
# ============================================================

EXPERIMENT_ID = "FRONT_BEHAVIOR_MODELS_001"

RANDOM_SEED = 42

MODEL_BEHAVIOR_CLASSES = [
    "NORMAL_SWIM",
    "PAIR_INTERACTION",
    "SHELTER_TRANSITION",
    "FEEDING",
]

MODEL_SAMPLE_STRIDE_SEC = 5.0
CV_BLOCK_SEC = 20.0
MAX_CV_SPLITS = 5

MIN_TRAINABLE_WINDOWS = 40
MIN_CLASSES = len(
    MODEL_BEHAVIOR_CLASSES
)

# Hard minimum required to attempt grouped CV.
MIN_WINDOWS_PER_CLASS = 5

# Scientific caution threshold only; does not stop execution.
RECOMMENDED_WINDOWS_PER_CLASS = 15

MODEL_COMPARISON_PATH = (
    RESULTS_DIR
    / "front_behavior_model_comparison.csv"
)

CV_PRED_PATH = (
    RESULTS_DIR
    / "front_behavior_cv_predictions.csv"
)

BEST_MODEL_PATH = (
    MODEL_DIR
    / "front_behavior_model.joblib"
)

MODEL_META_PATH = (
    MODEL_DIR
    / "front_behavior_model_metadata.json"
)

print(
    "Model classes:",
    MODEL_BEHAVIOR_CLASSES,
)

print(
    "Model sample stride:",
    MODEL_SAMPLE_STRIDE_SEC,
)

print(
    "CV block:",
    CV_BLOCK_SEC,
)

In [ ]:
# ============================================================
# 2. LOAD + JOIN FEATURES / LABELS — STRICT 4-CLASS FILTER
# ============================================================

feat = pd.read_csv(
    FEATURE_PATH
)

labels = pd.read_csv(
    LABEL_PATH
)

schema = json.loads(
    SCHEMA_PATH.read_text(
        encoding="utf-8"
    )
)

FEATURE_COLUMNS = schema[
    "feature_columns"
]

data = feat.merge(
    labels[
        [
            "window_id",
            "behavior_label",
            "label_certainty",
            "annotation_note",
        ]
    ],
    on="window_id",
    how="inner",
)

# Only manually CERTAIN labels are eligible.
data = data[
    data["label_certainty"]
    .astype(str)
    .str.upper()
    .eq("CERTAIN")
].copy()

# Final scientific decision: exactly four modeled behaviors.
data = data[
    data["behavior_label"]
    .isin(MODEL_BEHAVIOR_CLASSES)
].copy()

# Use non-overlapping 5 s starts for model evaluation.
data = data[
    np.isclose(
        np.mod(
            data["window_start_sec"],
            MODEL_SAMPLE_STRIDE_SEC,
        ),
        0.0,
        atol=1e-6,
    )
].copy()

data["cv_block_id"] = (
    data["video_id"]
    .astype(str)
    + ":B"
    + np.floor(
        data["window_start_sec"]
        / CV_BLOCK_SEC
    )
    .astype(int)
    .astype(str)
)

missing_features = [
    c
    for c in FEATURE_COLUMNS
    if c not in data.columns
]

if missing_features:
    raise ValueError(
        f"Missing feature columns: {missing_features}"
    )

data = (
    data.dropna(
        subset=(
            FEATURE_COLUMNS
            + ["behavior_label"]
        )
    )
    .reset_index(
        drop=True
    )
)

class_counts = (
    data["behavior_label"]
    .value_counts()
    .reindex(
        MODEL_BEHAVIOR_CLASSES,
        fill_value=0,
    )
)

trajectory_counts = (
    data.groupby(
        "behavior_label"
    )["trajectory_uid"]
    .nunique()
    .reindex(
        MODEL_BEHAVIOR_CLASSES,
        fill_value=0,
    )
)

group_counts_by_class = (
    data.groupby(
        "behavior_label"
    )["cv_block_id"]
    .nunique()
    .reindex(
        MODEL_BEHAVIOR_CLASSES,
        fill_value=0,
    )
)

print(
    "Trainable windows:",
    len(data),
)

print(
    "4-class counts:",
    class_counts.to_dict(),
)

print(
    "Unique trajectories/class:",
    trajectory_counts.to_dict(),
)

print(
    "CV blocks/class:",
    group_counts_by_class.to_dict(),
)

print(
    "Total CV groups:",
    data["cv_block_id"].nunique(),
)

missing_classes = [
    c
    for c in MODEL_BEHAVIOR_CLASSES
    if int(
        class_counts.loc[c]
    ) == 0
]

if missing_classes:
    raise RuntimeError(
        "MISSING_MODEL_CLASSES: "
        + ", ".join(
            missing_classes
        )
    )

if len(data) < MIN_TRAINABLE_WINDOWS:
    raise RuntimeError(
        "NOT_ENOUGH_LABELED_WINDOWS"
    )

if data["behavior_label"].nunique() != MIN_CLASSES:
    raise RuntimeError(
        "NOT_EXACTLY_FOUR_BEHAVIOR_CLASSES"
    )

too_small = {
    c: int(
        class_counts.loc[c]
    )
    for c
    in MODEL_BEHAVIOR_CLASSES
    if int(
        class_counts.loc[c]
    )
    < MIN_WINDOWS_PER_CLASS
}

if too_small:
    raise RuntimeError(
        "CLASS_TOO_SMALL_FOR_CV: "
        + str(
            too_small
        )
    )

caution = {
    c: int(
        class_counts.loc[c]
    )
    for c
    in MODEL_BEHAVIOR_CLASSES
    if int(
        class_counts.loc[c]
    )
    < RECOMMENDED_WINDOWS_PER_CLASS
}

if caution:
    print(
        "WARNING — small class counts:",
        caution,
    )

    print(
        "Interpret per-class and macro-F1 results cautiously. "
        "This warning is expected for FEEDING in the current GT set."
    )

In [ ]:
# ============================================================
# 3. MODEL DEFINITIONS + GROUPED CV WITH CLASS-COVERAGE CHECK
# ============================================================

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
import joblib


X = data[
    FEATURE_COLUMNS
].to_numpy(
    float
)

y = (
    data["behavior_label"]
    .astype(str)
    .to_numpy()
)

groups = (
    data["cv_block_id"]
    .astype(str)
    .to_numpy()
)


MODELS = {
    "MAJORITY_BASELINE":
        DummyClassifier(
            strategy="most_frequent"
        ),

    "LOGISTIC_REGRESSION":
        Pipeline(
            [
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "model",
                    LogisticRegression(
                        max_iter=3000,
                        class_weight="balanced",
                        random_state=RANDOM_SEED,
                    ),
                ),
            ]
        ),

    "RANDOM_FOREST":
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced_subsample",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            min_samples_leaf=2,
        ),
}


def _fold_has_all_classes(
    indices,
):
    observed = set(
        y[indices].tolist()
    )

    return (
        set(
            MODEL_BEHAVIOR_CLASSES
        )
        <= observed
    )


def _try_stratified_group_cv(
    n_splits,
):
    splitter = (
        StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=RANDOM_SEED,
        )
    )

    splits = list(
        splitter.split(
            X,
            y,
            groups,
        )
    )

    coverage = []

    for fold, (
        train_idx,
        test_idx,
    ) in enumerate(
        splits,
        start=1,
    ):
        coverage.append(
            {
                "fold": fold,
                "train_all_classes":
                    _fold_has_all_classes(
                        train_idx
                    ),
                "test_all_classes":
                    _fold_has_all_classes(
                        test_idx
                    ),
                "train_classes":
                    sorted(
                        set(
                            y[
                                train_idx
                            ].tolist()
                        )
                    ),
                "test_classes":
                    sorted(
                        set(
                            y[
                                test_idx
                            ].tolist()
                        )
                    ),
            }
        )

    all_ok = all(
        r[
            "train_all_classes"
        ]
        and
        r[
            "test_all_classes"
        ]
        for r
        in coverage
    )

    return (
        splits,
        coverage,
        all_ok,
    )


max_possible_splits = int(
    min(
        MAX_CV_SPLITS,
        group_counts_by_class.min(),
        data[
            "cv_block_id"
        ].nunique(),
    )
)

split_iter = None
coverage_records = None
split_name = None

for n_splits in range(
    max_possible_splits,
    1,
    -1,
):
    candidate_splits, candidate_coverage, ok = (
        _try_stratified_group_cv(
            n_splits
        )
    )

    if ok:
        split_iter = candidate_splits
        coverage_records = candidate_coverage
        split_name = (
            "StratifiedGroupKFold"
        )
        break

if split_iter is None:
    raise RuntimeError(
        "CV_CLASS_COVERAGE_FAILED — "
        "No grouped CV split from 2..MAX_CV_SPLITS "
        "contains all four behavior classes in every train and test fold. "
        "Add more labeled windows/groups, especially FEEDING."
    )

CV_COVERAGE = pd.DataFrame(
    coverage_records
)

print(
    "CV:",
    split_name,
    "splits:",
    len(
        split_iter
    ),
)

display(
    CV_COVERAGE
)

In [ ]:
# ============================================================
# 4. CROSS-VALIDATE MODELS
# ============================================================
all_predictions = []
comparison = []

labels_order = list(MODEL_BEHAVIOR_CLASSES)

for model_name, model in MODELS.items():
    fold_scores = []
    pred_records = []

    for fold, (train_idx, test_idx) in enumerate(split_iter, start=1):
        m = clone(model)
        m.fit(X[train_idx], y[train_idx])
        pred = m.predict(X[test_idx])

        fold_scores.append({
            "fold": fold,
            "accuracy": accuracy_score(y[test_idx], pred),
            "balanced_accuracy": balanced_accuracy_score(y[test_idx], pred),
            "macro_f1": f1_score(y[test_idx], pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y[test_idx], pred, average="weighted", zero_division=0),
        })

        for idx, p in zip(test_idx, pred):
            pred_records.append({
                "model": model_name,
                "fold": fold,
                "window_id": data.iloc[idx]["window_id"],
                "video_id": data.iloc[idx]["video_id"],
                "cv_block_id": data.iloc[idx]["cv_block_id"],
                "y_true": y[idx],
                "y_pred": p,
            })

    fs = pd.DataFrame(fold_scores)
    comparison.append({
        "model": model_name,
        "cv_method": split_name,
        "cv_splits": len(split_iter),
        "accuracy_mean": fs["accuracy"].mean(),
        "balanced_accuracy_mean": fs["balanced_accuracy"].mean(),
        "macro_f1_mean": fs["macro_f1"].mean(),
        "macro_f1_std": fs["macro_f1"].std(ddof=0),
        "weighted_f1_mean": fs["weighted_f1"].mean(),
    })
    all_predictions.extend(pred_records)

COMPARISON = pd.DataFrame(comparison).sort_values("macro_f1_mean", ascending=False).reset_index(drop=True)
CV_PRED = pd.DataFrame(all_predictions)

COMPARISON.to_csv(MODEL_COMPARISON_PATH, index=False)
CV_PRED.to_csv(CV_PRED_PATH, index=False)

CV_COVERAGE_PATH = RESULTS_DIR / "front_behavior_cv_fold_class_coverage.csv"
CV_COVERAGE.to_csv(CV_COVERAGE_PATH, index=False)

display(COMPARISON)
print("Saved:", MODEL_COMPARISON_PATH)

In [ ]:
# ============================================================
# 5. BEST MODEL — CONFUSION MATRIX + REPORT
# ============================================================
BEST_MODEL_NAME = str(COMPARISON.iloc[0]["model"])
best_pred = CV_PRED[CV_PRED["model"].eq(BEST_MODEL_NAME)].copy()

print("Best model:", BEST_MODEL_NAME)

report = classification_report(
    best_pred["y_true"],
    best_pred["y_pred"],
    labels=labels_order,
    output_dict=True,
    zero_division=0,
)
REPORT = pd.DataFrame(report).T
display(REPORT)

cm = confusion_matrix(
    best_pred["y_true"],
    best_pred["y_pred"],
    labels=labels_order,
    normalize="true",
)
CM = pd.DataFrame(cm, index=labels_order, columns=labels_order)
display(CM)

REPORT.to_csv(RESULTS_DIR / "front_behavior_best_model_classification_report.csv")
CM.to_csv(RESULTS_DIR / "front_behavior_best_model_confusion_matrix.csv")

plt.figure(figsize=(7,6))
plt.imshow(cm, aspect="auto")
plt.xticks(range(len(labels_order)), labels_order, rotation=45, ha="right")
plt.yticks(range(len(labels_order)), labels_order)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Normalized confusion matrix — {BEST_MODEL_NAME}")
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6. FIT FINAL MODEL + EXPORT FOR NOTEBOOK 18 / PI
# ============================================================
BEST_MODEL = clone(MODELS[BEST_MODEL_NAME])
BEST_MODEL.fit(X, y)

joblib.dump(BEST_MODEL, BEST_MODEL_PATH)

# Small inference benchmark on this computer.
sample = X[: min(len(X), 100)]
repeats = 200
t0 = time.perf_counter()
for _ in range(repeats):
    _ = BEST_MODEL.predict(sample)
elapsed = time.perf_counter() - t0
per_window_ms = (elapsed / (repeats * len(sample))) * 1000 if len(sample) else np.nan

meta = {
    "experiment_id": EXPERIMENT_ID,
    "best_model": BEST_MODEL_NAME,
    "selection_metric": "macro_f1_mean",
    "feature_columns": FEATURE_COLUMNS,
    "classes": list(MODEL_BEHAVIOR_CLASSES),
    "n_classes": int(len(MODEL_BEHAVIOR_CLASSES)),
    "class_counts": {c: int(class_counts.loc[c]) for c in MODEL_BEHAVIOR_CLASSES},
    "window_sec": schema["window_sec"],
    "step_sec": schema["step_sec"],
    "model_sample_stride_sec_for_cv": MODEL_SAMPLE_STRIDE_SEC,
    "cv_block_sec": CV_BLOCK_SEC,
    "cv_method": split_name,
    "cv_splits": len(split_iter),
    "trainable_windows": int(len(data)),
    "inference_ms_per_window_pc": float(per_window_ms),
    "track_id_note": "Track ID is not guaranteed biological identity.",
    "taxonomy_note": "LOW_ACTIVITY excluded because no confirmed GT windows were observed; UNCERTAIN/EXCLUDE are QC states.",
    "small_class_note": "FEEDING is currently a small class; interpret per-class and macro-F1 metrics cautiously.",
    "stress_note": "No stress label unless supported by biological ground truth.",
}
MODEL_META_PATH.write_text(json.dumps(meta, indent=2), encoding="utf-8")

print("Saved model:", BEST_MODEL_PATH)
print("Saved metadata:", MODEL_META_PATH)
print("Approx inference ms/window on current computer:", per_window_ms)

In [ ]:
# ============================================================
# 7. PI-FACING PREDICTION CONTRACT
# ============================================================
print("Realtime contract:")
print("- Input: latest 5 s cleaned trajectory features for one Track ID")
print("- Update cadence: every 1 s")
print("- Display fields:")
print("  Track ID")
print("  distance_last_1s_px")
print("  predicted behavior")
print("  confidence if model supports predict_proba")
print("- Track ID must be presented as tracker identity, not guaranteed biological identity.")

if hasattr(BEST_MODEL, "predict_proba"):
    proba = BEST_MODEL.predict_proba(X[:1])[0]
    classes = BEST_MODEL.classes_
    print("Example probabilities:", dict(zip(classes, map(float, proba))))

In [ ]:
# ============================================================
# 8. FINAL SUMMARY
# ============================================================

summary = {
    "experiment_id":
        EXPERIMENT_ID,

    "trainable_windows":
        int(
            len(data)
        ),

    "classes":
        list(
            MODEL_BEHAVIOR_CLASSES
        ),

    "n_classes":
        int(
            len(
                MODEL_BEHAVIOR_CLASSES
            )
        ),

    "class_counts":
        {
            c: int(
                class_counts.loc[c]
            )
            for c
            in MODEL_BEHAVIOR_CLASSES
        },

    "cv_method":
        split_name,

    "cv_splits":
        int(
            len(
                split_iter
            )
        ),

    "best_model":
        BEST_MODEL_NAME,

    "best_macro_f1_mean":
        float(
            COMPARISON.iloc[0][
                "macro_f1_mean"
            ]
        ),

    "best_balanced_accuracy_mean":
        float(
            COMPARISON.iloc[0][
                "balanced_accuracy_mean"
            ]
        ),

    "taxonomy_note": (
        "Four modeled behaviors only: NORMAL_SWIM, "
        "PAIR_INTERACTION, SHELTER_TRANSITION, FEEDING. "
        "LOW_ACTIVITY excluded because no confirmed GT windows "
        "were observed; UNCERTAIN/EXCLUDE are QC states."
    ),

    "small_class_warning": (
        "FEEDING has limited GT windows; interpret per-class "
        "and macro-F1 metrics cautiously."
    ),

    "outputs": [
        str(
            MODEL_COMPARISON_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        str(
            CV_PRED_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        str(
            CV_COVERAGE_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        str(
            BEST_MODEL_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        str(
            MODEL_META_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
    ],
}

(
    LOG_DIR
    / "summary.json"
).write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "FINAL SUMMARY"
)

print(
    json.dumps(
        summary,
        indent=2,
    )
)